In [1]:
# CELL 1: IMPORT
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import os

In [2]:
train_df = pd.read_csv("Dataset/train.csv")
val_df = pd.read_csv("Dataset/val.csv")
test_df = pd.read_csv("Dataset/test.csv")

In [3]:
# CELL 3: BUILD VOCAB

from collections import Counter

def tokenize(text):
    return text.lower().split()

counter = Counter()

for text in train_df["question"]:
    counter.update(tokenize(text))

for text in train_df["answer"]:
    counter.update(tokenize(text))

vocab = {
    "<pad>": 0,
    "<sos>": 1,
    "<eos>": 2,
    "<unk>": 3
}

for word, freq in counter.items():
    if freq >= 2:
        vocab[word] = len(vocab)

inv_vocab = {v: k for k, v in vocab.items()}

print("Vocab size:", len(vocab))

Vocab size: 80


In [4]:
# CELL 4: ENCODE FUNCTION

MAX_LEN = 20

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(t, vocab["<unk>"]) for t in tokens]

    ids = [vocab["<sos>"]] + ids + [vocab["<eos>"]]

    if len(ids) < MAX_LEN:
        ids += [vocab["<pad>"]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

In [5]:
# CELL 5: DATASET

class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)

        question = torch.tensor(encode(row["question"]))
        answer   = torch.tensor(encode(row["answer"]))

        return image, question, answer

In [6]:
# CELL 6: DATALOADER

train_dataset = VQADataset(train_df)
val_dataset   = VQADataset(val_df)

train_loader_a1 = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader      = DataLoader(val_dataset, batch_size=16)

In [7]:
# CELL 7: IMAGE ENCODER

import torchvision.models as models

class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        modules = list(resnet.children())[:-1]

        self.resnet = nn.Sequential(*modules)

    def forward(self, x):
        x = self.resnet(x)
        x = x.view(x.size(0), -1)
        return x

In [8]:
# CELL 8: QUESTION ENCODER

class QuestionEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)

        h = torch.cat((h[-2], h[-1]), dim=1)
        return h

In [9]:
# CELL 9: LSTM DECODER

class AnswerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden):
        x = self.embedding(x)        
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)           
        return out, hidden

In [10]:
# CELL 17: POSITIONAL ENCODING

import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        
        position = torch.arange(0, max_len).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer("pe", pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [11]:
# CELL 9B: TRANSFORMER DECODER (cho A2)

class TransformerAnswerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=2, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model)  # dùng lại class đã có ở Cell 17
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        # tgt: (B, L)
        # memory: (B, 1, d_model) — context từ image+question
        x = self.embedding(tgt)          # (B, L, d_model)
        x = self.pos_enc(x)              # (B, L, d_model)
        
        out = self.transformer_decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )                                # (B, L, d_model)
        
        out = self.fc(out)               # (B, L, vocab_size)
        return out

In [12]:
# CELL 10: FULL VQA MODEL

class VQAModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.img_encoder = ImageEncoder()
        self.q_encoder = QuestionEncoder(vocab_size)
        
        self.fc = nn.Linear(2048 + 512, 256)
        
        self.decoder = AnswerDecoder(vocab_size)
    
    def forward(self, image, question, answer):
    
        img_feat = self.img_encoder(image)
        q_feat = self.q_encoder(question)
        
        
        combined = torch.cat((img_feat, q_feat), dim=1)
        combined = self.fc(combined)
        
       
        h0 = combined.unsqueeze(0)   
        c0 = torch.zeros_like(h0)
        
        
        outputs, _ = self.decoder(answer[:, :-1], (h0, c0))
        
        return outputs

In [13]:
# CELL 11: TRAIN SETUP

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VQAModel(len(vocab)).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [14]:
#CELL 12:

def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss
    }, path)


def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
    print(f" Loaded checkpoint from epoch {checkpoint['epoch']}")
    
    return checkpoint["epoch"], checkpoint["loss"]

In [15]:
#CELL 13
import os
from tqdm import tqdm

EPOCHS = 10
start_epoch = 0

if not os.path.exists("best_model_a1.pth"):
    print(" Training model...")

    best_loss = float("inf")
    train_losses_a1 = []
    val_losses_a1 = []

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        total_loss = 0
        
        loop = tqdm(train_loader_a1, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for img, ques, ans in loop:
            img, ques, ans = img.to(device), ques.to(device), ans.to(device)
            
            optimizer.zero_grad()
            output = model(img, ques, ans)
            
            loss = criterion(
                output.reshape(-1, output.shape[-1]),
                ans[:, 1:].reshape(-1)
            )
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            loop.set_postfix(
                loss=loss.item(),
                avg_loss=total_loss/(loop.n+1)
            )

        avg_train_loss = total_loss / len(train_loader_a1)
        train_losses_a1.append(avg_train_loss)
        print(f"Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}")

        

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for img, ques, ans in val_loader:
                img, ques, ans = img.to(device), ques.to(device), ans.to(device)

                output = model(img, ques, ans)

                loss = criterion(
                    output.reshape(-1, output.shape[-1]),
                    ans[:, 1:].reshape(-1)
                )

                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        val_losses_a1.append(avg_val_loss)
        print(f"Epoch {epoch+1}, Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), "best_model_a1.pth")
            print(" Saved BEST model")

    print(" Training done")

else:
    print(" Model already exists, skip training")

 Model already exists, skip training


In [16]:
# CELL 14: LOAD MODEL 

import os

if os.path.exists("best_model_a1.pth"):
    model.load_state_dict(torch.load("best_model_a1.pth", map_location=device))
    model.to(device)
    model.eval()
    print(" Loaded trained model - NO NEED TO TRAIN")
else:
    print(" Model not found, please train first")

 Loaded trained model - NO NEED TO TRAIN


In [17]:
# CELL 15: GENERATE ANSWER

def generate_answer(model, image_path, question_text):
    model.eval()
    
    image = Image.open(image_path).convert("RGB")
    image = transforms.Resize((224, 224))(image)
    image = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
    ])(image).unsqueeze(0).to(device)
    
    question = torch.tensor([encode(question_text)]).to(device)
    
    with torch.no_grad():
        img_feat = model.img_encoder(image)
        q_feat = model.q_encoder(question)
        
        combined = torch.cat((img_feat, q_feat), dim=1)
        combined = model.fc(combined)
        
        h = combined.unsqueeze(0)
        c = torch.zeros_like(h)
        
        input_token = torch.tensor([[vocab["<sos>"]]]).to(device)
        
        result = []
        
        for _ in range(20):
            output, (h, c) = model.decoder(input_token, (h, c))
            
            token = output.argmax(-1)[:, -1]
            word = inv_vocab[token.item()]
            
            if word == "<eos>":
                break
            
            result.append(word)
            input_token = token.unsqueeze(1)
    
    return " ".join(result)

In [ ]:
#CELL 16:



import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io

upload = widgets.FileUpload(accept='image/*', multiple=False)

question_box = widgets.Text(
    value='Đây là món gì?',
    description='Question:',
    layout=widgets.Layout(width='500px')
)

button = widgets.Button(description="Generate Answer")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        
        if len(upload.value) == 0:
            print(" Hãy upload ảnh")
            return
        
        # FIX Ở ĐÂY
        file = upload.value[0]
        img_bytes = file['content']
        
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        
        display(image)
        
        question = question_box.value
        
        # lưu tạm
        temp_path = "temp.jpg"
        image.save(temp_path)
        
        answer = generate_answer(model, temp_path, question)
        
        print("\nQuestion:", question)
        print("Answer:", answer)

button.on_click(on_click)

display(upload, question_box, button, output)

#đây là món gì? 
#món này thường ăn nóng hay nguội?
#Đây thuộc loại món gì?
#Món này thường được đựng trong gì?
#Đây có phải món nước không?
#Món ăn này có màu gì?

FileUpload(value=(), accept='image/*', description='Upload')

Text(value='Đây là món gì?', description='Question:', layout=Layout(width='500px'))

Button(description='Generate Answer', style=ButtonStyle())

Output()

: 

In [19]:
# CELL 17: POSITIONAL ENCODING

import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        
        position = torch.arange(0, max_len).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer("pe", pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [20]:
# CELL 17.5: LABEL MAP

labels = sorted(train_df["label_key"].unique())

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

NUM_CLASSES = len(labels)

print("Num classes:", NUM_CLASSES)

Num classes: 8


In [21]:
# CELL 18: VQA MODEL A2 — Transformer Decoder (FIXED)

class VQAModel_A2(nn.Module):
    def __init__(self, vocab_size, d_model=256):
        super().__init__()
        
        self.img_encoder = ImageEncoder()
        self.q_encoder = QuestionEncoder(vocab_size)
        
        # Project fusion vector → d_model, dùng làm memory cho decoder
        self.fc_memory = nn.Linear(2048 + 512, d_model)
        
        self.decoder = TransformerAnswerDecoder(vocab_size, d_model=d_model)
    
    @staticmethod
    def make_causal_mask(sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return mask
    
    def forward(self, image, question, answer):
      
        img_feat = self.img_encoder(image)     
        q_feat   = self.q_encoder(question)     
        
        # Fusion → memory
        combined = torch.cat((img_feat, q_feat), dim=1)   
        memory   = self.fc_memory(combined)               
        memory   = memory.unsqueeze(1)                   
        
        tgt = answer[:, :-1]                              
        
        tgt_mask = self.make_causal_mask(tgt.size(1), tgt.device)
        
        tgt_key_padding_mask = (tgt == 0)                
        
        outputs = self.decoder(tgt, memory, tgt_mask, tgt_key_padding_mask)
        return outputs  

In [22]:
#CELL 19:

from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def load_image(path):
    image = Image.open(path).convert("RGB")
    image = transform(image)
    return image

In [23]:
# CELL 20 

import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 1. CREATE LABEL MAP

labels = sorted(train_df["label_key"].unique())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
NUM_CLASSES = len(labels)

print("Num classes:", NUM_CLASSES)


# 2. DATASET A2 

class VQADataset_A2(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image    = load_image(row["image_path"])
        question = torch.tensor(encode(row["question"]))
        answer   = torch.tensor(encode(row["answer"]))   # ← đổi lại thành answer
        return image, question, answer


# 3. DATALOADER 
train_dataset_a2 = VQADataset_A2(train_df)
val_dataset_a2 = VQADataset_A2(val_df)

train_loader_a2 = DataLoader(
    train_dataset_a2,
    batch_size=8,  
    shuffle=True
)

val_loader_a2 = DataLoader(
    val_dataset_a2,
    batch_size=8,
    shuffle=False
)


# 4. MODEL + OPTIMIZER
model_a2 = VQAModel_A2(len(vocab)).to(device) 

optimizer = torch.optim.Adam(
    model_a2.parameters(),
    lr=1e-4
)

criterion = torch.nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

# 5. DEBUG SHAPE
img, ques, label = next(iter(train_loader_a2))

print("DEBUG SHAPE:")
print("img:", img.shape)
print("ques:", ques.shape)
print("label:", label.shape)


# 6. TRAINING + VALIDATION
EPOCHS = 10
best_loss = float("inf")

train_losses_a2 = []
val_losses_a2 = []

torch.cuda.empty_cache() 

for epoch in range(EPOCHS):

    #TRAIN 
    model_a2.train()
    total_loss = 0

    loop = tqdm(train_loader_a2, desc=f"[A2] Epoch {epoch+1}/{EPOCHS}")

    for img, ques, ans in loop:
        img  = img.to(device)
        ques = ques.to(device)
        ans  = ans.to(device)

        optimizer.zero_grad()
        output = model_a2(img, ques, ans)  

        loss = criterion(
            output.reshape(-1, output.shape[-1]),
            ans[:, 1:].reshape(-1)          
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_loader_a2)
    train_losses_a2.append(avg_train_loss)

    print(f"[A2] Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}")


    model_a2.eval()
    val_loss = 0

    with torch.no_grad():
        for img, ques, ans in val_loader_a2:       
            img  = img.to(device)
            ques = ques.to(device)
            ans  = ans.to(device)

            output = model_a2(img, ques, ans)      
            loss = criterion(
                output.reshape(-1, output.shape[-1]),
                ans[:, 1:].reshape(-1)             
            )

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader_a2)
    val_losses_a2.append(avg_val_loss)

    print(f"[A2] Epoch {epoch+1} - Val Loss: {avg_val_loss:.4f}")


    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model_a2.state_dict(), "best_model_a2.pth")
        print("Saved BEST A2 model")


print("TRAIN A2 DONE")

Num classes: 8
DEBUG SHAPE:
img: torch.Size([8, 3, 224, 224])
ques: torch.Size([8, 20])
label: torch.Size([8, 20])


[A2] Epoch 1/10:   1%|          | 6/495 [00:10<13:41,  1.68s/it, loss=3.58]


KeyboardInterrupt: 

In [24]:
#CELL 21 : LOAD A2 MODEL

if os.path.exists("best_model_a2.pth"):
    model_a2.load_state_dict(torch.load("best_model_a2.pth"))
    model_a2.eval()
    print("✅ Loaded A2 model")

✅ Loaded A2 model


In [25]:
# CELL 22: GENERATE ANSWER A2 — Transformer Decoder

def generate_answer_a2(model, image_path, question_text, max_len=20):
    model.eval()

    image = load_image(image_path).unsqueeze(0).to(device) 
    question = torch.tensor([encode(question_text)]).to(device)

    with torch.no_grad():
        img_feat = model.img_encoder(image)
        q_feat   = model.q_encoder(question)

        combined = torch.cat((img_feat, q_feat), dim=1)
        memory   = model.fc_memory(combined).unsqueeze(1)  

        generated = torch.tensor([[vocab["<sos>"]]], device=device)  

        result = []

        for _ in range(max_len):
            tgt_mask = VQAModel_A2.make_causal_mask(generated.size(1), device)

            output = model.decoder(generated, memory, tgt_mask) 
            next_token = output[:, -1, :].argmax(dim=-1)          
            word = inv_vocab[next_token.item()]

            if word == "<eos>":
                break

            result.append(word)
            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)

    return " ".join(result)

In [ ]:
# CELL 23: DEMO A2 (INTERACTIVE)

import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io

upload = widgets.FileUpload(accept='image/*', multiple=False)

question_box = widgets.Text(
    value='Đây là món gì?',
    description='Question:',
    layout=widgets.Layout(width='500px')
)

button = widgets.Button(description="Generate Answer (A2)")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        
        if len(upload.value) == 0:
            print("❌ Hãy upload ảnh")
            return
        
        file = upload.value[0]
        img_bytes = file['content']
        
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        display(image)
        
        question = question_box.value
        
        temp_path = "temp.jpg"
        image.save(temp_path)
        
    
        answer = generate_answer_a2(model_a2, temp_path, question)
        
        print("\nQuestion:", question)
        print("Answer:", answer)

button.on_click(on_click)

display(upload, question_box, button, output) 

#đây là món gì? 
#món này thường ăn nóng hay nguội?
#Đây thuộc loại món gì?
#Món này thường được đựng trong gì?
#Đây có phải món nước không?

FileUpload(value=(), accept='image/*', description='Upload')

Text(value='Đây là món gì?', description='Question:', layout=Layout(width='500px'))

Button(description='Generate Answer (A2)', style=ButtonStyle())

Output()

In [27]:
# CELL 26: CÀI THƯ VIỆN

%pip install rouge-score bert-score nltk -q

import nltk
nltk.download('punkt')

You should consider upgrading via the 'd:\HK6\DeepLearning\VQACK\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [28]:
# CELL 27: EVALUATION PIPELINE (A1 vs A2)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rouge_lib
from bert_score import score as bert_score_fn

smoother = SmoothingFunction().method1
scorer_rouge = rouge_lib.RougeScorer(['rougeL'], use_stemmer=False)

def evaluate_model(generate_fn, model, test_df, model_name="Model"):
    exact_matches = []
    bleu_scores   = []
    rougeL_scores = []
    preds_all     = []
    refs_all      = []

    print(f"\n🔍 Đánh giá {model_name} trên {len(test_df)} mẫu test...")

    for _, row in test_df.iterrows():
        pred = generate_fn(model, row["image_path"], row["question"])
        ref  = row["answer"].lower().strip()
        pred = pred.lower().strip()

        preds_all.append(pred)
        refs_all.append(ref)

        # Exact match
        exact_matches.append(int(pred == ref))

        # BLEU
        bleu = sentence_bleu(
            [ref.split()],
            pred.split(),
            smoothing_function=smoother
        )
        bleu_scores.append(bleu)

        # ROUGE-L
        rouge = scorer_rouge.score(ref, pred)
        rougeL_scores.append(rouge["rougeL"].fmeasure)

    # BERTScore (tính 1 lần cho toàn bộ)
    print("  Đang tính BERTScore...")
    P, R, F1 = bert_score_fn(preds_all, refs_all, lang="vi", verbose=False)

    results = {
        "model":        model_name,
        "exact_match":  sum(exact_matches) / len(test_df),
        "bleu":         sum(bleu_scores)   / len(test_df),
        "rougeL":       sum(rougeL_scores) / len(test_df),
        "bertscore_f1": F1.mean().item(),
        "preds":        preds_all,
        "refs":         refs_all,
    }

    print(f"   Exact Match : {results['exact_match']:.2%}")
    print(f"   BLEU        : {results['bleu']:.4f}")
    print(f"   ROUGE-L     : {results['rougeL']:.4f}")
    print(f"   BERTScore F1: {results['bertscore_f1']:.4f}")

    return results

In [ ]:
# CELL 28: CHẠY ĐÁNH GIÁ A1 vs A2

results_a1 = evaluate_model(generate_answer,    model,    test_df, "A1 (LSTM decoder)")
results_a2 = evaluate_model(generate_answer_a2, model_a2, test_df, "A2 (Transformer decoder)")


🔍 Đánh giá A1 (LSTM decoder) trên 1683 mẫu test...
  Đang tính BERTScore...


In [ ]:
# CELL 29: BẢNG SO SÁNH + BIỂU ĐỒ

import matplotlib.pyplot as plt
import pandas as pd

# Bảng so sánh
metrics = ["exact_match", "bleu", "rougeL", "bertscore_f1"]
labels  = ["Exact Match", "BLEU", "ROUGE-L", "BERTScore F1"]

df_compare = pd.DataFrame([
    {
        "Model": r["model"],
        "Exact Match": f"{r['exact_match']:.2%}",
        "BLEU":        f"{r['bleu']:.4f}",
        "ROUGE-L":     f"{r['rougeL']:.4f}",
        "BERTScore F1":f"{r['bertscore_f1']:.4f}",
    }
    for r in [results_a1, results_a2]
])

print("\n===== BẢNG SO SÁNH A1 vs A2 =====")
print(df_compare.to_string(index=False))

# Biểu đồ
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle("A1 (LSTM) vs A2 (Transformer) — Evaluation", fontsize=13)

colors = ["#4C72B0", "#DD8452"]

for ax, metric, label in zip(axes, metrics, labels):
    vals = [results_a1[metric], results_a2[metric]]
    bars = ax.bar(["A1", "A2"], vals, color=colors, width=0.5)
    ax.set_title(label, fontsize=11)
    ax.set_ylim(0, max(vals) * 1.3 + 0.01)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("eval_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Đã lưu biểu đồ: eval_comparison.png")